In [0]:
from pyspark.sql import functions as f

### History

In [0]:

df = (spark.read.format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .option("multiLine", "true")   # permite quebras de linha dentro de aspas
      .option("escape", '"')         # trata "" dentro do texto como aspas literais
      .load("/Volumes/workspace/asn/metrics/history.csv"))
      

df.display()

In [0]:
shape = (df.count(), len(df.columns))
print(shape)

#### GENDER

In [0]:
df.select("gender").distinct().display()

#### SeniorCitizen

In [0]:
df.select("SeniorCitizen").distinct().display()

#### Partner

In [0]:
df.select("Partner").distinct().display()

#### Dependents


In [0]:


df_clientes_dependentes = (df
               .groupBy("Dependents")
               .count()
               )

df_clientes_dependentes.display()

#### tenure

In [0]:
df_clientes_tenure = (df
               .groupBy("tenure")
               .count()
               )

df_clientes_tenure.display()

#### PhoneService


In [0]:
df_clientes_PhoneService = (df
               .groupBy("PhoneService")
               .count()
               )

df_clientes_PhoneService.display()

#### MultipleLines

In [0]:
df_clientes_MultipleLines = (df
               .groupBy("MultipleLines")
               .count()
               )

df_clientes_MultipleLines.display()

#### InternetService

In [0]:
df_clientes_InternetService = (df
               .groupBy("InternetService")
               .count()
               )

df_clientes_InternetService.display()

#### OnlineSecurity

In [0]:
df_clientes_OnlineSecurity = (df
               .groupBy("OnlineSecurity")
               .count()
               )

df_clientes_OnlineSecurity.display()

#### OnlineBackup

In [0]:
df_clientes_OnlineBackup = (df
               .groupBy("OnlineBackup")
               .count()
               )

df_clientes_OnlineBackup.display()

#### DeviceProtection


In [0]:
df_clientes_DeviceProtection = (df
               .groupBy("DeviceProtection")
               .count()
               )

df_clientes_DeviceProtection.display()

#### TechSupport


In [0]:
df_clientes_TechSupport = (df
               .groupBy("TechSupport")
               .count()
               )

df_clientes_TechSupport.display()

#### StreamingTV


In [0]:
df_clientes_StreamingTV = (df
               .groupBy("StreamingTV")
               .count()
               )

df_clientes_StreamingTV.display()

#### StreamingMovies


In [0]:
df_clientes_StreamingMovies= (df
               .groupBy("StreamingMovies")
               .count()
               )

df_clientes_StreamingMovies.display()

#### Contract


In [0]:
df_clientes_Contract= (df
               .groupBy("Contract")
               .count()
               )

df_clientes_Contract.display()

#### PaperlessBilling

In [0]:
df_clientes_PaperlessBilling= (df
               .groupBy("PaperlessBilling")
               .count()
               )

df_clientes_PaperlessBilling.display()

#### PaymentMethod

In [0]:
df_clientes_PaymentMethod = (df
               .groupBy("PaymentMethod")
               .count()
               )

df_clientes_PaymentMethod.display()

#### MonthlyCharges

In [0]:
df_clientes_MonthlyCharges = (df
               .groupBy("MonthlyCharges")
               .count()
               )

df_clientes_MonthlyCharges.display()

#### TotalCharges

In [0]:
df_clientes_TotalCharges = (df
               .groupBy("TotalCharges")
               .count()
               )

df_clientes_TotalCharges.display()

#### Churn

In [0]:
df_clientes_Churn = (df
               .groupBy("Churn")
               .count()
               )

df_clientes_Churn.display()

#### CustomerFeedback

In [0]:
df.select("CustomerFeedback").distinct().display()

#### MonthlyIncome


In [0]:
df_clientes_MonthlyIncome = (df
               .groupBy("MonthlyIncome")
               .count()
               )

df_clientes_MonthlyIncome.display()

### PLN

In [0]:
df_pandas = df.select("CustomerFeedback", "Churn").toPandas()

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

v = TfidfVectorizer(
    ngram_range=(1, 2),     # palavras soltas e pares
    min_df=20,              # ignora termos que aparecem em menos de 20 textos
    max_df=0.8,             # ignora termos que aparecem em mais de 80% dos textos
    stop_words="english",   # remove "the", "and", "is"...
    max_features=100)       # fica só com os 100 termos mais frequentes

X = v.fit_transform(df_pandas["CustomerFeedback"])

tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=["tfidf_" + t.replace(" ", "_") for t in v.get_feature_names_out()],
    index=df_pandas.index)

df_final = pd.concat([df_pandas.drop(columns="CustomerFeedback"), tfidf_df], axis=1)

In [0]:
df_final.display()

In [0]:
from sklearn.decomposition import TruncatedSVD
import pandas as pd

svd = TruncatedSVD(n_components=5, random_state=42)
Z = svd.fit_transform(X)                       # X = matriz do TF-IDF

# 1) As colunas novas, uma linha por cliente
svd_df = pd.DataFrame(Z, columns=[f"svd_{i+1}" for i in range(5)], index=df.index)
svd_df.head()

# 2) O que cada coluna significa: peso de cada palavra em cada componente
pesos = pd.DataFrame(svd.components_, index=svd_df.columns,
                     columns=v.get_feature_names_out())    # v = seu TfidfVectorizer
pesos.loc["svd_2"].sort_values()

In [0]:
svd_df.display()

In [0]:
pesos.display()

In [0]:
shape = (df_final.count(), len(df_final.columns))
print(shape)

In [0]:
import pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df = df_final

media = df.groupby("Churn").mean(numeric_only=True).T
media.index = media.index.str.replace("tfidf_", "").str.replace("_", " ")
media["diff"] = media["Yes"] - media["No"]

N = 12
top = pd.concat([media.nlargest(N, "diff"), media.nsmallest(N, "diff")]).sort_values("diff")

AZUL, VERMELHO, TXT, TXT2 = "#2a78d6", "#e34948", "#1a1a19", "#52514e"
fig, ax = plt.subplots(figsize=(9, 8), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
cores = [VERMELHO if d > 0 else AZUL for d in top["diff"]]
ax.barh(top.index, top["diff"], color=cores, height=0.62)
ax.axvline(0, color="#8a8984", lw=1)
for y, (t, d) in enumerate(top["diff"].items()):
    ax.text(d + (0.002 if d > 0 else -0.002), y, f"{d:+.3f}", va="center",
            ha="left" if d > 0 else "right", fontsize=9, color=TXT2)
ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index, fontsize=10, color=TXT)
lim = top["diff"].abs().max() * 1.18
ax.set_xlim(-lim, lim)
for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.tick_params(axis="x", colors=TXT2, labelsize=9); ax.tick_params(axis="y", length=0)
ax.grid(axis="x", color="#e6e5e0", lw=0.8); ax.set_axisbelow(True)
ax.set_xlabel("Diferença na média de TF-IDF  (churn − não churn)", color=TXT2, fontsize=10)
fig.text(0.04, 0.955, "Quais palavras distinguem quem saiu de quem ficou",
         fontsize=15, fontweight="bold", color=TXT)
fig.text(0.04, 0.925, f"Top {N} termos de cada lado · {int((df.Churn=='Yes').sum())} clientes com churn vs {int((df.Churn=='No').sum())} sem",
         fontsize=10, color=TXT2)
fig.text(0.80, 0.895, "● Mais forte em quem deu churn", color=VERMELHO, fontsize=10, ha="center")
fig.text(0.50, 0.895, "● Mais forte em quem ficou", color=AZUL, fontsize=10, ha="center")
fig.subplots_adjust(left=0.22, right=0.97, top=0.87, bottom=0.10)
fig.savefig("/Volumes/workspace/asn/metrics/palavras_churn.png", dpi=160, facecolor=fig.get_facecolor())
print(top[["No","Yes","diff"]].round(3).tail(6)); print(top[["No","Yes","diff"]].round(3).head(6))

### Análise de sentimento

In [0]:
# %pip install vaderSentiment textblob

In [0]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

df = df_final

analyzer = SentimentIntensityAnalyzer()
df["vader_compound"] = df_pandas["CustomerFeedback"].map(lambda t: analyzer.polarity_scores(t)["compound"])
df["textblob_polarity"] = df_pandas["CustomerFeedback"].map(lambda t: TextBlob(t).sentiment.polarity)

In [0]:
df.display()



### Inference

In [0]:

df_Inference = (spark.read.format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .option("multiLine", "true")   # permite quebras de linha dentro de aspas
      .option("escape", '"')         # trata "" dentro do texto como aspas literais
      .load("/Volumes/workspace/asn/metrics/inference.csv"))
      

df_Inference.display()

In [0]:
shape = (df_Inference.count(), len(df_Inference.columns))
print(shape)

In [0]:
df_Inference.printSchema()


#### Gender

In [0]:
df_Inference.columns

In [0]:
df_Inference_gender = (df_Inference
               .groupBy("gender")
               .count()
               )

df_Inference_gender.display()

#### SeniorCitizen

In [0]:
df_Inference_SeniorCitizen = (df_Inference
               .groupBy("SeniorCitizen")
               .count()
               )

df_Inference_SeniorCitizen.display()

#### Partner

In [0]:
df_Inference_Partner = (df_Inference
               .groupBy("Partner")
               .count()
               )

df_Inference_Partner.display()

#### Dependents

In [0]:
df_Inference_Dependents = (df_Inference
               .groupBy("Dependents")
               .count()
               )

df_Inference_Dependents.display()

#### tenure

In [0]:
df_Inference_tenure = (df_Inference
               .groupBy("tenure")
               .count()
               )

df_Inference_tenure.display()

#### PhoneService

In [0]:
df_Inference_PhoneService = (df_Inference
               .groupBy("PhoneService")
               .count()
               )

df_Inference_PhoneService.display()

#### InternetService

In [0]:
df_Inference_InternetService = (df_Inference
               .groupBy("InternetService")
               .count()
               )

df_Inference_InternetService.display()

#### OnlineSecurity

In [0]:
df_Inference_OnlineSecurity = (df_Inference
               .groupBy("OnlineSecurity")
               .count()
               )

df_Inference_OnlineSecurity.display()

#### OnlineBackup

In [0]:
df_Inference_OnlineBackup = (df_Inference
               .groupBy("OnlineBackup")
               .count()
               )

df_Inference_OnlineBackup.display()



#### DeviceProtection

In [0]:
df_Inference_DeviceProtection = (df_Inference
               .groupBy("DeviceProtection")
               .count()
               )

df_Inference_DeviceProtection.display()



#### TechSupport

In [0]:
df_Inference_TechSupport = (df_Inference
               .groupBy("TechSupport")
               .count()
               )

df_Inference_TechSupport.display()



#### StreamingTV

In [0]:
df_Inference_StreamingTV = (df_Inference
               .groupBy("StreamingTV")
               .count()
               )

df_Inference_StreamingTV.display()



#### StreamingMovies

In [0]:
df_Inference_StreamingMovies = (df_Inference
               .groupBy("StreamingMovies")
               .count()
               )

df_Inference_StreamingMovies.display()



#### Contract

In [0]:
df_Inference_Contract = (df_Inference
               .groupBy("Contract")
               .count()
               )

df_Inference_Contract.display()



#### PaperlessBilling

In [0]:
df_Inference_PaperlessBilling = (df_Inference
               .groupBy("PaperlessBilling")
               .count()
               )

df_Inference_PaperlessBilling.display()



#### PaymentMethod

In [0]:
df_Inference_PaymentMethod = (df_Inference
               .groupBy("PaymentMethod")
               .count()
               )

df_Inference_PaymentMethod.display()



#### MonthlyCharges

In [0]:
df_Inference_MonthlyCharges = (df_Inference
               .groupBy("MonthlyCharges")
               .count()
               )

df_Inference_MonthlyCharges.display()



#### TotalCharges

In [0]:
df_Inference_TotalCharges = (df_Inference
               .groupBy("TotalCharges")
               .count()
               )

df_Inference_TotalCharges.display()



In [0]:
df_Inference_TotalCharges = (df_Inference
               .groupBy("TotalCharges")
               .count()
               )

df_Inference_TotalCharges.filter(f.col("TotalCharges") == "null").display()



In [0]:
df_Inference_TotalCharges = (df_Inference
               .groupBy("TotalCharges")
               .count()
               )

df_Inference_TotalCharges.filter(f.col("TotalCharges").isNull()).display()



In [0]:
df_Inference_TotalCharges = (df_Inference
               .groupBy("TotalCharges")
               .count()
               )

df_Inference_TotalCharges.filter(f.col("TotalCharges") == "0").display()



#### CustomerFeedback


In [0]:
df_Inference.filter(f.col("CustomerFeedback").isNull()).display()



In [0]:
df_Inference.filter(f.col("CustomerFeedback") == "null").display()



#### MonthlyIncome

In [0]:
df_Inference_MonthlyIncome = (df_Inference
               .groupBy("MonthlyIncome")
               .count()
               )

df_Inference_MonthlyIncome.display()

In [0]:
df_Inference_MonthlyIncome = (df_Inference
               .groupBy("MonthlyIncome")
               .count()
               )

df_Inference_MonthlyIncome.filter(f.col("MonthlyIncome").isNull()).display()

In [0]:
df_Inference.shape

### PLN

In [0]:
df_pandas = df_Inference.select("CustomerFeedback").toPandas()

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

v = TfidfVectorizer(
    ngram_range=(1, 2),     # palavras soltas e pares
    min_df=20,              # ignora termos que aparecem em menos de 20 textos
    max_df=0.8,             # ignora termos que aparecem em mais de 80% dos textos
    stop_words="english",   # remove "the", "and", "is"...
    max_features=100)       # fica só com os 100 termos mais frequentes

X = v.fit_transform(df_pandas["CustomerFeedback"])

tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=["tfidf_" + t.replace(" ", "_") for t in v.get_feature_names_out()],
    index=df_pandas.index)

df_final = pd.concat([df_pandas.drop(columns="CustomerFeedback"), tfidf_df], axis=1)

In [0]:
df_final.display()